# EF データ抽出スクリプト
各 Patient_ID において Date が古い行のみを対象に、text_data 内の「EF」の前後 10 文字を抽出します。

出力ファイル：`~/Desktop/EF.10word.csv`

In [13]:
import sqlite3
import pandas as pd
import os

# DB ファイルのパス
db_path = '/Users/muna/Hana_research/data/db/Hana_Research.db'

# 接続
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 各 Patient_ID ごとに Date が古い行のみを選択
query = """
SELECT * FROM Freedocument
WHERE (Patients_ID, Date) IN (
    SELECT Patients_ID, MIN(Date) 
    FROM Freedocument
    GROUP BY Patients_ID
);
"""

cursor.execute(query)
df = pd.read_sql_query(query, conn)

print(f"取得した行数: {len(df)}")
print(f"カラム: {list(df.columns)}")

cursor.close()
conn.close()

取得した行数: 3822
カラム: ['Patients_ID', 'Date', 'document_type', 'text_data', 'Study_ID']


In [14]:
# EF の前後 10 文字を抽出する関数
def extract_EF_surrounding(text):
    """
    テキスト内の 'EF' を検索し、前後 10 文字を抽出する。
    'EF' が見つからない場合は None を返す。
    """
    if pd.isna(text) or text == '':
        return None
    
    import re
    # 'EF' の位置を取得
    matches = list(re.finditer(r'EF', text))
    
    if not matches:
        return None
    
    extracted_texts = []
    
    for match in matches:
        start_pos = match.start()
        end_pos = match.end()
        
        # 前後 10 文字を確保
        start = max(0, start_pos - 10)
        end = min(len(text), end_pos + 10)
        
        extracted_text = text[start:end]
        extracted_texts.append(extracted_text)
    
    return extracted_texts if len(extracted_texts) > 1 else extracted_texts[0]

In [15]:
# EF 前後 10 文字を抽出
df['EF_10_word'] = df['text_data'].apply(extract_EF_surrounding)

# EF が見つかった行のみを選択
df_ef = df[df['EF_10_word'].notna()]

print(f"'EF' を含む行数: {len(df_ef)}")

# 不要なカラムを削除（必要に応じて保持する場合はコメントアウト）
# df_output = df_ef[['Patients_ID', 'Date', 'document_type', 'EF_10_word']]
df_output = df_ef[['Patients_ID', 'Date', 'document_type', 'EF_10_word']]

df_output

'EF' を含む行数: 1290


,Patients_ID,Date,document_type,EF_10_word
1,150002,2015-04-18,初診時サマリー平成26年4月,不全症候群\nHF/pEF。利尿剤内服にてコン
5,150006,2015-04-18,初診時サマリー平成26年4月,"VDd 54mm, EF 30%(diffu"
8,150009,2015-04-18,初診時サマリー平成26年4月,"preserved EF, moderate"
17,150018,2015-04-09,初診時サマリー,弁生体弁\n置換術後、EF＜20%）、非持続性
19,150020,2015-04-18,初診時サマリー平成26年4月,性心不全（病因不明、EF 30-40%、Dd
...,...,...,...,...
3813,251069,2025-06-03,訪問診療初診時サマリー2025年5月,0日、訪問診療開始。EF 61％(2025/
3814,251276,2025-05-09,訪問診療初診時サマリー2025年4月,",Ds 26mm, EF 56％, 左室局所"
3815,251313,2025-09-29,高橋 春子様 訪問診療初診時サマリー2025年9月,"[ 慢性心不全(HFpEF)、重症大動脈弁狭窄, ,Ds 25mm, EF 75％, 左室局所]"
3817,251462,2025-08-28,吉田 悠治様 訪問診療初診時サマリー2025年8月,"[# 慢性心不全HFpEF, 狭心症、冠動脈形, synergy-, EF 57％, IVST]"


In [16]:
# CSV ファイルを保存
output_path = os.path.expanduser('~/Desktop/EF.10word.csv')

# 出力カラムを指定
output_columns = ['Patients_ID', 'Date', 'document_type', 'EF_10_word']
df_output.to_csv(output_path, index=False, columns=output_columns)

print(f"CSV ファイルを保存しました: {output_path}")
print(f"保存された行数: {len(df_output)}")

CSV ファイルを保存しました: /Users/muna/Desktop/EF.10word.csv
保存された行数: 1290
